# Neural networks: composition must earn its complexity

**Question.** Does a small multilayer model improve cultivar prediction enough to justify a more difficult optimization and operations contract?

A neural network composes the same preactivation and activation ideas used in the single-neuron models. We will compare it with logistic regression, inspect its training trace and confusion matrix, and state what a student backpropagation implementation must verify.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.evaluation import comparison_table
from dsm_project.experiments import fit_supervised_candidate

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)

comparison = comparison_table(
    dataset,
    config,
    task_name="classification",
    model_names=("logistic_regression", "neural_network"),
)
display(comparison)

## Composition and expressive power

For layer $\ell$,

$$
z^{(\ell)}=W^{(\ell)}a^{(\ell-1)}+b^{(\ell)},
\qquad
a^{(\ell)}=\phi^{(\ell)}(z^{(\ell)}).
$$

Without nonlinear activations, composing affine layers collapses to one affine map. Nonlinearity lets hidden units build piecewise or curved representations. Backpropagation applies the chain rule from the loss toward earlier layers, producing gradients for every weight and bias.

Greater expressiveness creates obligations: initialization, scaling, optimization convergence, regularization, architecture selection, calibration, resource measurement, and reproducible randomness. A neural network is not modern merely because it is deep.

In [ ]:
result = fit_supervised_candidate(
    dataset,
    config,
    task_name="classification",
    model_name="neural_network",
)
network = result.estimator.named_steps["mlpclassifier"]

print("Implementation:", result.model_name)
print(result.metric_name, result.metric_value)
print("Iterations:", network.n_iter_)
print("Final training loss:", network.loss_)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(network.loss_curve_, color="#e7298a")
axes[0].set(title="Optimization trace", xlabel="optimizer iteration", ylabel="training loss")

ConfusionMatrixDisplay.from_predictions(
    result.observed,
    result.predictions,
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("Held-out confusion matrix")
plt.tight_layout();

## What the plots do and do not show

A falling training loss is an optimizer diagnostic, not generalization evidence. The confusion matrix is held-out evidence for one split, not a guarantee across future batches or subpopulations. Compare the network with linear, tree, neighbor and majority baselines under the same validation design. Measure wall time, artifact size and prediction latency when those properties matter to the user.

## Conclusion, limits, and student implementation

The network should remain only if its added complexity improves the declared evidence or enables behavior simpler models cannot provide. On a small tabular dataset, a strong linear or tree model may be the more professional choice.

Build the educational network from explicit layers of neurons first: forward pass, activation, output loss, chain-rule error propagation, gradients and updates. Verify gradients numerically on a tiny deterministic network before scaling. Then explain how matrix operations implement the same computation efficiently. Register the student implementation and compare it with the trusted reference through the package boundary.